<a href="https://colab.research.google.com/github/luisfelipetrj99/ai-data-science-portfolio/blob/main/projects/enterprise-text-to-sql-rag-agent/enterprise_text_to_sql_agent_spanish_version_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Agente Empresarial Text-to-SQL con RAG

## Descripción General del Proyecto

Este proyecto implementa un agente de inteligencia artificial tipo **Text-to-SQL corporativo**, capaz de transformar preguntas en lenguaje natural en consultas SQL ejecutables sobre una base de datos empresarial.

La solución combina:
- Large Language Models (LLMs)
- Retrieval-Augmented Generation (RAG)
- Embeddings y búsqueda semántica
- Indexación vectorial con FAISS
- Recuperación de metadata
- Prompt engineering
- Generación automática de SQL

El proyecto simula un entorno empresarial real donde usuarios de negocio pueden consultar información utilizando lenguaje natural y obtener consultas SQL automáticamente.

---

## Descripción del Conjunto de Datos

El conjunto de datos utilizado para este reto contiene:

### Metadata de la Base de Datos
Para cada tabla:
- Schema de la tabla
- Definición de columnas
- Descripción funcional
- Contexto de negocio

### Ejemplos de Consultas
Cada tabla incluye:
- Queries SQL de ejemplo
- Descripción de las consultas
- Casos de uso de negocio

Toda esta información se utiliza como contexto para alimentar el agente Text-to-SQL.

---

## Objetivos del Proyecto

Los principales objetivos son:

1. Construir un asistente corporativo Text-to-SQL
2. Reducir alucinaciones mediante RAG
3. Mejorar la calidad de generación SQL
4. Optimizar el consumo de tokens
5. Diseñar una arquitectura escalable para asistentes analíticos empresariales

---

## Arquitectura de la Solución

La solución implementa múltiples enfoques:

### 1. Baseline Text-to-SQL
Estrategia directa donde todo el schema se inyecta al modelo.

### 2. Text-to-SQL basado en RAG
Arquitectura donde únicamente se recupera el contexto relevante utilizando embeddings y búsqueda semántica con FAISS.

### 3. Recuperación de Metadata tipo Key-Value
Estrategia adicional para optimizar la recuperación contextual a nivel tabla.

---

## Temas Cubiertos

Este notebook cubre:

- Procesamiento de Lenguaje Natural (NLP)
- Sistemas Text-to-SQL
- Retrieval-Augmented Generation (RAG)
- Bases de datos vectoriales
- Búsqueda semántica
- Prompt engineering
- Embeddings
- Indexación con FAISS
- Orquestación de LLMs
- Recuperación de metadata
- Generación de SQL
- Optimización de tokens
- Sistemas empresariales de IA
- Agentes de IA
- Information Retrieval

---

## Skills y Competencias Demostradas

### Machine Learning & AI
- Desarrollo de sistemas NLP
- Desarrollo de aplicaciones con LLMs
- Sistemas de recuperación semántica
- Diseño de agentes inteligentes

### Data Engineering
- Pipelines de generación SQL
- Procesamiento de metadata
- Sistemas de recuperación de schemas

### MLOps & AI Engineering
- Arquitectura modular de IA
- Pipelines RAG
- Diseño escalable de prompts

### Software Engineering
- Desarrollo en Python
- Diseño reutilizable de pipelines
- Arquitectura enterprise-ready

---

## Casos de Uso Empresariales

Este tipo de solución puede utilizarse para:

- Self-service analytics
- BI copilots
- Asistentes empresariales de datos
- Analítica CRM
- Reportes financieros automáticos
- Monitoreo operativo
- Analítica de marketing
- Soporte analítico para negocio

---

## Posibles Mejoras Futuras

Algunas mejoras potenciales incluyen:

- Guardrails de validación SQL
- Loops de retroalimentación
- Arquitectura multi-agente
- Optimización automática de queries
- Fine-tuning especializado
- Memoria conversacional
- Control de acceso por roles
- Generación automática de dashboards
- Integración con streaming analytics

---

## Tecnologías Utilizadas

- Python
- OpenAI API
- FAISS
- Pandas
- NumPy
- Jupyter Notebook
- Embeddings
- Pipelines RAG

---

## Acknowledgment

El conjunto de datos, así como la idea base de este proyecto, pertenecen al Colegio de Matemáticas Bourbaki y fueron desarrollados como parte del curso de NLP Avanzado.

Este repositorio presenta una implementación y extensión personal del proyecto con fines académicos, de aprendizaje y portafolio profesional.


In [ ]:
!pip install faiss-cpu

In [ ]:
!pip -q install openai langchain-core

In [ ]:
import pandas as pd
import getpass
from openai import OpenAI
import numpy as np

#api_key = getpass.getpass("Introduce tu OpenAI API Key: ")
#client = OpenAI(api_key=api_key)

url = "https://raw.githubusercontent.com/luisfelipetrj99/ai-data-science-portfolio/refs/heads/main/projects/enterprise-text-to-sql-rag-agent/data/sql_dataset_bourbaki.json"

df = pd.read_json(url)

# Ver primeras filas
print(df.head())

# Ver estructura
print(df.info())

                                                         users  \
description  Almacena información de cuentas de clientes, i...   
schema       id INT PRIMARY KEY, first_name VARCHAR(50), la...   
examples     [{'description': 'Buscar un usuario por correo...   

                                                    categories  \
description  Categorías de productos para organizar el catá...   
schema       id INT PRIMARY KEY, parent_id INT, name VARCHA...   
examples     [{'description': 'Obtener todas las categorías...   

                                                      products  \
description  Detalles sobre artículos a la venta, precios, ...   
schema       id INT PRIMARY KEY, category_id INT, name VARC...   
examples     [{'description': 'Listar productos inactivos o...   

                                                        orders  \
description  Datos transaccionales de alto nivel para compr...   
schema       id INT PRIMARY KEY, user_id INT, order_date TI...   
example

In [ ]:
df['users']

,users
description,"Almacena información de cuentas de clientes, i..."
schema,"id INT PRIMARY KEY, first_name VARCHAR(50), la..."
examples,[{'description': 'Buscar un usuario por correo...


In [ ]:
#Tomo como ejemplo la tabla de users para ver el contenido
for i, row in enumerate(df['users']):
    print(f"\n--- Registro {i} ---\n")
    print(row)


--- Registro 0 ---

Almacena información de cuentas de clientes, incluyendo detalles de contacto y fecha de registro.

--- Registro 1 ---

id INT PRIMARY KEY, first_name VARCHAR(50), last_name VARCHAR(50), email VARCHAR(100) UNIQUE, phone VARCHAR(20), created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP

--- Registro 2 ---

[{'description': 'Buscar un usuario por correo electrónico.', 'sql': "SELECT * FROM users WHERE email = 'cliente@example.com';"}, {'description': 'Contar nuevos usuarios en el último mes.', 'sql': "SELECT COUNT(*) FROM users WHERE created_at >= CURRENT_DATE - INTERVAL '1 month';"}, {'description': 'Buscar usuarios sin número de teléfono.', 'sql': 'SELECT id, first_name, last_name FROM users WHERE phone IS NULL;'}]


## Baseline del modelo:

Para poder tener un baseline de este modelo convertire cada tabla en su propio documento incluyendo los datos proporcionados de cada tabla en el json y con esto entrenaré el primer modelo

In [ ]:
from langchain_core.documents import Document
GENERATION_MODEL="gpt-4.1-mini"

In [ ]:
def build_table_document(table_name, description, schema, examples):

    examples_text = ""

    for i, ex in enumerate(examples, 1):
        examples_text += f"\n{i}. Description: {ex.get('description')}\n"
        examples_text += f"   SQL: {ex.get('query')}\n"

    doc_text = f"""
Table Name: {table_name}

Description:
{description}

Schema:
{schema}

Examples:
{examples_text}
"""

    return doc_text.strip()

In [ ]:
documents = []

for table in df.columns:

    description = df.loc["description", table]
    schema = df.loc["schema", table]
    examples = df.loc["examples", table]

    content = build_table_document(
        table_name=table,
        description=description,
        schema=schema,
        examples=examples
    )

    doc = Document(
        page_content=content,
        metadata={
            "table_name": table,
            "doc_type": "table_full"
        }
    )

    documents.append(doc)

In [ ]:
documents

[Document(metadata={'table_name': 'users', 'doc_type': 'table_full'}, page_content='Table Name: users\n\nDescription:\nAlmacena información de cuentas de clientes, incluyendo detalles de contacto y fecha de registro.\n\nSchema:\nid INT PRIMARY KEY, first_name VARCHAR(50), last_name VARCHAR(50), email VARCHAR(100) UNIQUE, phone VARCHAR(20), created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP\n\nExamples:\n\n1. Description: Buscar un usuario por correo electrónico.\n   SQL: None\n\n2. Description: Contar nuevos usuarios en el último mes.\n   SQL: None\n\n3. Description: Buscar usuarios sin número de teléfono.\n   SQL: None'),
 Document(metadata={'table_name': 'categories', 'doc_type': 'table_full'}, page_content='Table Name: categories\n\nDescription:\nCategorías de productos para organizar el catálogo jerárquicamente.\n\nSchema:\nid INT PRIMARY KEY, parent_id INT, name VARCHAR(100), description TEXT\n\nExamples:\n\n1. Description: Obtener todas las categorías principales (sin padre).\n   SQL

In [ ]:
#Consulte con chat gpt un promp mejorado basado en como pretendo hacer el baseline
SYSTEM_PROMPT = (
    "Eres un asistente experto en generación de consultas SQL.\n\n"

    "Tu tarea es generar consultas SQL basándote únicamente en el contexto proporcionado.\n"
    "El contexto incluye:\n"
    "- Nombre de las tablas\n"
    "- Descripción de las tablas\n"
    "- Schema (columnas y tipos)\n"
    "- Ejemplos de queries\n\n"

    "Reglas estrictas:\n"
    "1. SOLO puedes usar las tablas mencionadas en el contexto.\n"
    "2. SOLO puedes usar las columnas definidas en el schema.\n"
    "3. NO inventes tablas, columnas ni relaciones.\n"
    "4. Si la información no es suficiente para responder, di: 'No hay suficiente información para generar la consulta'.\n"
    "5. Genera únicamente consultas SQL válidas.\n"
    "6. NO incluyas explicaciones, comentarios ni texto adicional.\n"
    "7. NO uses lenguaje natural, SOLO SQL.\n\n"

    "Buenas prácticas:\n"
    "- Usa alias cuando sea necesario.\n"
    "- Usa JOINs correctamente si la consulta requiere múltiples tablas.\n"
    "- Usa funciones agregadas (COUNT, SUM, AVG, etc.) cuando aplique.\n"
    "- Aplica filtros (WHERE) de forma correcta.\n\n"

    "Formato de salida:\n"
    "Devuelve únicamente la consulta SQL en texto plano.\n"
)

In [ ]:
import os
import getpass

from openai import OpenAI
from google.colab import userdata
api_key = getpass.getpass("Introduce tu OpenAI API Key: ")
client=OpenAI(api_key=api_key)


Introduce tu OpenAI API Key: ··········


In [ ]:
#Concateno texto
contexto_completo = "\n\n" + ("\n\n" + "="*80 + "\n\n").join(
    [doc.page_content for doc in documents]
)

print("Longitud del contexto:", len(contexto_completo))
print("\nVista previa del contexto:\n")
print(contexto_completo[:2000])

Longitud del contexto: 4440

Vista previa del contexto:



Table Name: users

Description:
Almacena información de cuentas de clientes, incluyendo detalles de contacto y fecha de registro.

Schema:
id INT PRIMARY KEY, first_name VARCHAR(50), last_name VARCHAR(50), email VARCHAR(100) UNIQUE, phone VARCHAR(20), created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP

Examples:

1. Description: Buscar un usuario por correo electrónico.
   SQL: None

2. Description: Contar nuevos usuarios en el último mes.
   SQL: None

3. Description: Buscar usuarios sin número de teléfono.
   SQL: None


Table Name: categories

Description:
Categorías de productos para organizar el catálogo jerárquicamente.

Schema:
id INT PRIMARY KEY, parent_id INT, name VARCHAR(100), description TEXT

Examples:

1. Description: Obtener todas las categorías principales (sin padre).
   SQL: None

2. Description: Listar todas las subcategorías para el ID de categoría 5.
   SQL: None


Table Name: products

Description:
Detalles so

In [ ]:
pregunta= "Dame todos los usuarios registrados con su correo electrónico."
response = client.chat.completions.create(
        model=GENERATION_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": f"Contexto:\n{contexto_completo}\n\nPregunta: {pregunta}"
            }
        ],
        temperature=0
    )


In [ ]:
sql_generado = response.choices[0].message.content

print(sql_generado)

SELECT id, first_name, last_name, email FROM users;


Para una consulta sencilla y sin filtros complejos, el enfoque baseline parece comportarse correctamente. En este caso, el modelo logró identificar la tabla adecuada y seleccionar los campos necesarios, incluyendo el correo electrónico solicitado para todos los usuarios.

La consulta SQL generada fue:

SELECT id, first_name, last_name, email FROM users

Como siguiente paso, se evaluará el modelo con:
1. Una pregunta que no debería poder responder debido a la falta de contexto suficiente en la base de datos.
2. Una consulta más compleja para analizar su capacidad de generar queries con mayor nivel de razonamiento y lógica SQL.

In [ ]:
pregunta = "¿Cuál es el salario promedio de los empleados de la empresa?"
response = client.chat.completions.create(
        model=GENERATION_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": f"Contexto:\n{contexto_completo}\n\nPregunta: {pregunta}"
            }
        ],
        temperature=0
    )
sql_generado = response.choices[0].message.content

print(sql_generado)

No hay suficiente información para generar la consulta


No contesto una pregunta a la que le falta contexto

In [ ]:
pregunta =  "¿Cuál es el total de dinero gastado por cada usuario en pedidos, considerando solo aquellos que tienen más de 2 órdenes?"
response = client.chat.completions.create(
        model=GENERATION_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": f"Contexto:\n{contexto_completo}\n\nPregunta: {pregunta}"
            }
        ],
        temperature=0
    )
sql_generado = response.choices[0].message.content

print(sql_generado)

SELECT 
    u.id AS user_id,
    u.first_name,
    u.last_name,
    SUM(o.total_amount) AS total_spent,
    COUNT(o.id) AS total_orders
FROM 
    users u
JOIN 
    orders o ON u.id = o.user_id
GROUP BY 
    u.id, u.first_name, u.last_name
HAVING 
    COUNT(o.id) > 2;


Para una consulta más compleja, el modelo también fue capaz de generar correctamente la query utilizando la estructura documental proporcionada como contexto.

Aunque este enfoque baseline resulta funcional, aún presenta limitaciones importantes. La solución depende en gran medida de la capacidad del LLM para interpretar correctamente todo el contexto enviado y tomar decisiones adecuadas a partir de él. Este enfoque puede funcionar con una cantidad reducida de información; sin embargo, conforme aumenta el número de tablas, schemas y ejemplos disponibles, el contexto crece considerablemente.

Al escalar el sistema, esto provocaría:
- Mayor consumo de tokens.
- Incremento en el costo por consulta a la API.
- Mayor tiempo de respuesta.
- Incremento en la probabilidad de alucinaciones o generación de consultas incorrectas.

Con el baseline ya establecido, el siguiente paso consiste en mejorar la arquitectura incorporando embeddings y Retrieval-Augmented Generation (RAG).

Para ello, la información será dividida en dos componentes diferentes, tal como fue sugerido durante el curso:

1. Un conjunto de documentos que contendrá el schema y la descripción de cada tabla, junto con metadata indicando a qué tabla pertenece.
2. Un key-value store que almacenará la información completa asociada a cada tabla.

Esta arquitectura permitirá recuperar únicamente el contexto más relevante antes de generar la consulta SQL, haciendo que el sistema sea más escalable, eficiente y preciso.

## Modelo Final

In [ ]:
import os
import getpass
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver


#Key valyue store
table_store = {}

for table_name in df.columns:
    table_store[table_name] = {
        "table_name": table_name,
        "description": df.loc["description", table_name],
        "schema": df.loc["schema", table_name],
        "examples": df.loc["examples", table_name]
    }

In [ ]:
print(table_store["users"]["description"])

Almacena información de cuentas de clientes, incluyendo detalles de contacto y fecha de registro.


In [ ]:
#Documento con metadata
documents = []

for table_name in df.columns:
    description = df.loc["description", table_name]

    content = f"""
Table Name: {table_name}

Description:
{description}
""".strip()

    doc = Document(
        page_content=content,
        metadata={
            "table_name": table_name
        }
    )

    documents.append(doc)

In [ ]:
documents

[Document(metadata={'table_name': 'users'}, page_content='Table Name: users\n\nDescription:\nAlmacena información de cuentas de clientes, incluyendo detalles de contacto y fecha de registro.'),
 Document(metadata={'table_name': 'categories'}, page_content='Table Name: categories\n\nDescription:\nCategorías de productos para organizar el catálogo jerárquicamente.'),
 Document(metadata={'table_name': 'products'}, page_content='Table Name: products\n\nDescription:\nDetalles sobre artículos a la venta, precios, vinculación de categorías e inventario.'),
 Document(metadata={'table_name': 'orders'}, page_content='Table Name: orders\n\nDescription:\nDatos transaccionales de alto nivel para compras de clientes, estado del pedido y costo total.'),
 Document(metadata={'table_name': 'order_items'}, page_content='Table Name: order_items\n\nDescription:\nTabla de unión que detalla productos específicos comprados en un pedido, cantidades y precios cerrados.'),
 Document(metadata={'table_name': 'revi

Procedo a definir la adquitectura basada en FAISS

In [ ]:
import numpy as np
import faiss
from openai import OpenAI
from google.colab import userdata

api_key = userdata.get("OPENAI_TOKEN")
client = OpenAI(api_key=api_key)



In [ ]:
#Parámetros globales:

EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIM=1536

#Modelo generacion
GENERATION_MODEL="gpt-4.1-mini"
#Folder
data_folder='content/data/'
supported_ext='.pdf'
#RAG Retrieval
TOP_K=3
CHUNK_SIZE=500
CHUNK_OVERLAP=50



In [ ]:
SYSTEM_PROMPT = (
    "Eres un asistente experto en generación de consultas SQL.\n\n"

    "Tu tarea es generar consultas SQL basándote únicamente en el contexto proporcionado.\n"
    "El contexto incluye:\n"
    "- Nombre de las tablas\n"
    "- Descripción de las tablas\n"
    "- Schema (columnas y tipos)\n"
    "- Ejemplos de queries\n\n"

    "Reglas estrictas:\n"
    "1. SOLO puedes usar las tablas mencionadas en el contexto.\n"
    "2. SOLO puedes usar las columnas definidas en el schema.\n"
    "3. NO inventes tablas, columnas ni relaciones.\n"
    "4. Si la información no es suficiente para responder, di: 'No hay suficiente información para generar la consulta'.\n"
    "5. Genera únicamente consultas SQL válidas.\n"
    "6. NO incluyas explicaciones, comentarios ni texto adicional.\n"
    "7. NO uses lenguaje natural, SOLO SQL.\n"
    "8. Prefiere apoyarte en los patrones de los ejemplos proporcionados.\n\n"

    "Formato de salida:\n"
    "Devuelve únicamente la consulta SQL en texto plano.\n"
)

In [ ]:
#FUNCION PARA HCER LOS EMBEDDINGS

def get_embedding(text: str, model: str = EMBEDDING_MODEL):
    response = client.embeddings.create(
        model=model,
        input=text
    )
    return response.data[0].embedding

In [ ]:
document_texts = [doc.page_content for doc in documents]
document_metadata = [doc.metadata for doc in documents]

embeddings_docs = []

for text in document_texts:
    emb = get_embedding(text)
    embeddings_docs.append(emb)

embeddings_docs = np.array(embeddings_docs, dtype="float32")

print("Shape embeddings:", embeddings_docs.shape)

Shape embeddings: (8, 1536)


In [ ]:
EMBEDDING_DIM = embeddings_docs.shape[1]

index = faiss.IndexFlatL2(EMBEDDING_DIM)
index.add(embeddings_docs)

print(f"Índice FAISS creado. Vectores indexados: {index.ntotal}")

Índice FAISS creado. Vectores indexados: 8


In [ ]:
### Funciones principales del modelo Retrieval-Augmented Generation (RAG)


# Recupera las tablas más relevantes utilizando búsqueda semántica
def retrieve_table_context(question: str, k: int = TOP_K, verbose: bool = True):
    """
    Recupera los documentos más relevantes para una pregunta en lenguaje natural
    utilizando búsqueda por similitud semántica con FAISS.

    La función transforma la pregunta del usuario en un embedding y posteriormente
    realiza una búsqueda en el índice vectorial para identificar los documentos
    más cercanos semánticamente.

    Parámetros
    ----------
    question : str
        Pregunta realizada por el usuario en lenguaje natural.

    k : int, opcional
        Número de documentos a recuperar.
        El valor por defecto está definido por TOP_K.

    verbose : bool, opcional
        Si es True, imprime información de depuración incluyendo
        los documentos recuperados y su metadata.

    Retorna
    -------
    list
        Lista de documentos recuperados ordenados por similitud semántica.
    """

    # Genera el embedding de la pregunta del usuario
    query_vec = np.array([get_embedding(question)], dtype="float32")

    # Realiza la búsqueda de similitud en el índice FAISS
    D, I = index.search(query_vec, min(k, index.ntotal))

    # Recupera los documentos correspondientes
    retrieved_docs = [documents[i] for i in I[0]]

    # Impresión opcional para inspeccionar los documentos recuperados
    if verbose:
        print("\n" + "=" * 60)
        print(f"Pregunta: {question}")
        print(f"\nDocumentos recuperados (top-{k}):")

        for j, doc in enumerate(retrieved_docs):
            print(f"\n[{j+1}] table_name = {doc.metadata['table_name']}")
            print(doc.page_content[:500])

    return retrieved_docs


# Recupera el contexto completo desde el key-value store
def get_full_context_from_store(retrieved_docs, table_store):
    """
    Recupera la información completa asociada a cada tabla
    desde el key-value store utilizando la metadata del documento.

    Después de identificar las tablas más relevantes mediante
    búsqueda semántica, esta función obtiene toda la información
    contextual necesaria para construir el prompt final.

    Parámetros
    ----------
    retrieved_docs : list
        Lista de documentos recuperados mediante FAISS.

    table_store : dict
        Diccionario que contiene la información completa de cada tabla.

    Retorna
    -------
    list
        Lista con el contexto completo de las tablas recuperadas.
    """

    full_contexts = []

    # Utiliza el nombre de la tabla como llave para recuperar
    # toda la información asociada
    for doc in retrieved_docs:
        table_name = doc.metadata["table_name"]
        full_contexts.append(table_store[table_name])

    return full_contexts


# Construye el contexto final que será enviado al LLM
def build_prompt_context(full_contexts):
    """
    Construye el contexto estructurado que será enviado al modelo LLM.

    El contexto incluye:
    - Nombre de la tabla
    - Descripción
    - Schema
    - Ejemplos de consultas SQL

    Esta estructura ayuda al modelo a generar consultas SQL
    más precisas y contextualizadas.

    Parámetros
    ----------
    full_contexts : list
        Lista que contiene la información completa de las tablas recuperadas.

    Retorna
    -------
    str
        Contexto final formateado para ser utilizado dentro del prompt.
    """

    parts = []

    # Recorre cada tabla recuperada
    for ctx in full_contexts:

        examples_text = ""

        # Formatea los ejemplos SQL asociados a cada tabla
        for i, ex in enumerate(ctx["examples"], 1):
            examples_text += (
                f"\n{i}. Description: {ex.get('description')}\n"
                f"   SQL: {ex.get('query')}\n"
            )

        # Construye la representación estructurada de cada tabla
        table_text = f"""
Table Name: {ctx['table_name']}

Description:
{ctx['description']}

Schema:
{ctx['schema']}

Examples:
{examples_text}
""".strip()

        parts.append(table_text)

    # Une todos los contextos recuperados en un solo bloque
    return "\n\n" + ("-" * 80 + "\n\n").join(parts)

In [ ]:
## Pipeline completo RAG
def rag_query_sql(question: str, k: int = TOP_K, verbose: bool = True):
    """
    Ejecuta el pipeline completo de Retrieval-Augmented Generation (RAG)
    para generar consultas SQL a partir de preguntas en lenguaje natural.

    Flujo del pipeline
    ------------------
    1. Genera el embedding de la pregunta del usuario.
    2. Recupera los documentos más relevantes mediante FAISS.
    3. Utiliza la metadata de los documentos recuperados para consultar
       el key-value store con la información completa de cada tabla.
    4. Construye el contexto final que será enviado al LLM.
    5. Genera la consulta SQL utilizando el modelo de lenguaje.

    Parámetros
    ----------
    question : str
        Pregunta del usuario en lenguaje natural.

    k : int, opcional
        Número de documentos relevantes a recuperar desde el índice FAISS.
        El valor por defecto está definido por TOP_K.

    verbose : bool, opcional
        Si es True, imprime información útil para depuración e inspección,
        incluyendo el contexto enviado al modelo, la respuesta generada y
        el número total de tokens utilizados.

    Retorna
    -------
    str
        Consulta SQL generada por el modelo.
    """

    # Paso 1: Recuperación semántica
    # Busca los documentos más relevantes para la pregunta del usuario
    # utilizando embeddings y similitud vectorial en FAISS.
    retrieved_docs = retrieve_table_context(question, k=k, verbose=verbose)

    # Paso 2: Consulta al key-value store
    # A partir de la metadata de los documentos recuperados,
    # se obtiene la información completa de las tablas relevantes.
    full_contexts = get_full_context_from_store(retrieved_docs, table_store)

    # Paso 3: Construcción del contexto final
    # Formatea la información recuperada para incluirla dentro del prompt.
    contexto_rag = build_prompt_context(full_contexts)

    # Construye el prompt final combinando el contexto recuperado
    # con la pregunta original del usuario.
    prompt_usuario = f"Contexto:\n{contexto_rag}\n\nPregunta: {question}"

    # Paso 4: Generación de SQL
    # Envía el prompt al LLM para generar la consulta SQL final.
    response = client.chat.completions.create(
        model=GENERATION_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt_usuario}
        ],
        temperature=0
    )

    # Extrae la respuesta generada por el modelo
    sql_output = response.choices[0].message.content.strip()

    # Impresión opcional para depuración e interpretación del resultado
    if verbose:
        print("\n" + "=" * 60)
        print("CONTEXTO FINAL ENVIADO AL MODELO:")
        print(contexto_rag[:3000])  # Vista previa del contexto

        print("\n" + "=" * 60)
        print(f"RESPUESTA ({GENERATION_MODEL}):")
        print(sql_output)

        print(f"\nTokens usados: {response.usage.total_tokens}")

    return sql_output

In [ ]:
#Pregunta 1
sql_1 = rag_query_sql("Dame todos los usuarios registrados con su correo electrónico.")


Pregunta: Dame todos los usuarios registrados con su correo electrónico.

Documentos recuperados (top-3):

[1] table_name = users
Table Name: users

Description:
Almacena información de cuentas de clientes, incluyendo detalles de contacto y fecha de registro.

[2] table_name = shipping_details
Table Name: shipping_details

Description:
Información logística, números de seguimiento y direcciones de entrega vinculadas a un pedido.

[3] table_name = reviews
Table Name: reviews

Description:
Calificaciones de clientes y comentarios escritos para productos específicos.

CONTEXTO FINAL ENVIADO AL MODELO:


Table Name: users

Description:
Almacena información de cuentas de clientes, incluyendo detalles de contacto y fecha de registro.

Schema:
id INT PRIMARY KEY, first_name VARCHAR(50), last_name VARCHAR(50), email VARCHAR(100) UNIQUE, phone VARCHAR(20), created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP

Examples:

1. Description: Buscar un usuario por correo electrónico.
   SQL: None

2. Descr

In [ ]:
#Pregunta sin contexto suficiente:
sql_2 = rag_query_sql("¿Cuál es el salario promedio de los empleados de la empresa?")


Pregunta: ¿Cuál es el salario promedio de los empleados de la empresa?

Documentos recuperados (top-3):

[1] table_name = promotions
Table Name: promotions

Description:
Códigos de descuento, porcentajes de oferta y rangos de fechas válidos para promociones de la tienda.

[2] table_name = users
Table Name: users

Description:
Almacena información de cuentas de clientes, incluyendo detalles de contacto y fecha de registro.

[3] table_name = orders
Table Name: orders

Description:
Datos transaccionales de alto nivel para compras de clientes, estado del pedido y costo total.

CONTEXTO FINAL ENVIADO AL MODELO:


Table Name: promotions

Description:
Códigos de descuento, porcentajes de oferta y rangos de fechas válidos para promociones de la tienda.

Schema:
id INT PRIMARY KEY, code VARCHAR(50) UNIQUE, discount_percentage DECIMAL(5, 2), start_date DATE, end_date DATE, is_active BOOLEAN

Examples:

1. Description: Verificar si un código de descuento está activo y es válido hoy.
   SQL: None

In [ ]:
#Pregunta mas complicada
sql_3 = rag_query_sql("¿Cuál es el total de dinero gastado por cada usuario en pedidos, considerando solo aquellos que tienen más de 2 órdenes?")



Pregunta: ¿Cuál es el total de dinero gastado por cada usuario en pedidos, considerando solo aquellos que tienen más de 2 órdenes?

Documentos recuperados (top-3):

[1] table_name = orders
Table Name: orders

Description:
Datos transaccionales de alto nivel para compras de clientes, estado del pedido y costo total.

[2] table_name = order_items
Table Name: order_items

Description:
Tabla de unión que detalla productos específicos comprados en un pedido, cantidades y precios cerrados.

[3] table_name = shipping_details
Table Name: shipping_details

Description:
Información logística, números de seguimiento y direcciones de entrega vinculadas a un pedido.

CONTEXTO FINAL ENVIADO AL MODELO:


Table Name: orders

Description:
Datos transaccionales de alto nivel para compras de clientes, estado del pedido y costo total.

Schema:
id INT PRIMARY KEY, user_id INT, order_date TIMESTAMP, status VARCHAR(50), total_amount DECIMAL(10, 2)

Examples:

1. Description: Obtener pedidos pendientes para 

En conclusión, el modelo basado en RAG resulta considerablemente más eficiente y escalable que el enfoque baseline inicial. A diferencia del modelo base, esta arquitectura no depende exclusivamente de la capacidad del LLM para interpretar grandes cantidades de contexto, sino que utiliza embeddings y búsqueda semántica con FAISS para recuperar únicamente la información más relevante antes de generar la consulta SQL.

Este enfoque permite:
- Reducir significativamente el ruido dentro del prompt.
- Disminuir el consumo de tokens y el costo asociado a las llamadas de API.
- Mejorar la precisión contextual de las respuestas.
- Reducir la probabilidad de alucinaciones.
- Escalar de mejor manera conforme aumenta el número de tablas, schemas y ejemplos disponibles.

Además, la separación entre retrieval semántico y generación de SQL hace que la arquitectura sea más modular, mantenible y cercana a soluciones reales utilizadas en entornos empresariales de NLP y sistemas basados en LLMs.

Finalmente, este proyecto demuestra cómo técnicas modernas como embeddings, vector search y Retrieval-Augmented Generation pueden integrarse para construir asistentes Text-to-SQL más robustos, eficientes y preparados para escenarios enterprise.